# 📊 AI-Assisted Trading Risk Manager
A simplified notebook covering: **Sentiment → Risk Models → Portfolio Optimization → Monte Carlo**

**Install dependencies first:**
```bash
pip install yfinance transformers torch numpy scipy statsmodels plotly arch
```

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# --- CONFIG ---
TICKERS   = ['AAPL', 'MSFT', 'XOM', 'GS', 'JPM']
START     = '2020-01-01'
END       = '2024-12-31'
RISK_FREE = 0.05          # annual risk-free rate
print('✅ Imports OK')

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.
The following compiled module files exist, but seem incompatible
with with either python 'cpython-314' or the platform 'win32':

  * _multiarray_umath.cp312-win_amd64.lib
  * _multiarray_umath.cp312-win_amd64.pyd

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python 3.14 from "d:\Romain\Projects\Finance\02_Risk Analysis\ai-quant-risk-engine\.venv\Scripts\python.exe"
  * The NumPy version is: "2.4.6"

and make sure that they are the versions you expect.

Please carefully study the information and documentation linked above.
This is unlikely to be a NumPy issue but will be caused by a bad install
or environment 

---
## Phase 1 — Financial Sentiment Engine (FinBERT)
Run NLP on headlines to produce Bullish / Bearish / Neutral scores.

In [ ]:
from transformers import pipeline

# Load FinBERT once (downloads ~420 MB on first run)
sentiment_pipe = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    return_all_scores=True
)

# Sample headlines — replace / extend with real feed
headlines = [
    "Apple beats earnings expectations, raises guidance",
    "Oil prices tumble on demand fears amid global slowdown",
    "Fed signals two more rate hikes this year",
    "Goldman Sachs upgrades Microsoft to strong buy",
    "JPMorgan warns of credit losses in commercial real estate",
]

results = sentiment_pipe(headlines, top_k=None)

rows = []
for headline, scores in zip(headlines, results):
    score_dict = {s['label']: s['score'] for s in scores}
    dominant   = max(score_dict, key=score_dict.get)
    rows.append({
        'headline':  headline,
        'positive':  round(score_dict.get('positive', 0), 3),
        'negative':  round(score_dict.get('negative', 0), 3),
        'neutral':   round(score_dict.get('neutral',  0), 3),
        'sentiment': dominant
    })

sentiment_df = pd.DataFrame(rows)
display(sentiment_df)

# Aggregate: single portfolio-level sentiment score (-1 … +1)
portfolio_sentiment = (
    sentiment_df['positive'].mean() - sentiment_df['negative'].mean()
)
print(f'\n📰 Portfolio sentiment score: {portfolio_sentiment:+.3f}')


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22337.32it/s]


,headline,positive,negative,neutral,sentiment
0,"Apple beats earnings expectations, raises guid...",0.914,0.024,0.063,positive
1,Oil prices tumble on demand fears amid global ...,0.015,0.965,0.019,negative
2,Fed signals two more rate hikes this year,0.506,0.251,0.244,positive
3,Goldman Sachs upgrades Microsoft to strong buy,0.926,0.036,0.037,positive
4,JPMorgan warns of credit losses in commercial ...,0.023,0.958,0.019,negative



📰 Portfolio sentiment score: +0.030


---
## Phase 2 — Risk Models
### 2a · Download price data & compute returns

In [ ]:
raw    = yf.download(TICKERS, start=START, end=END, auto_adjust=True)['Close']
prices = raw.dropna()
rets   = prices.pct_change().dropna()
print(f'Downloaded {len(prices)} days for {len(TICKERS)} tickers.')
prices.tail(3)

[*********************100%***********************]  5 of 5 completed

Downloaded 1257 days for 5 tickers.


Ticker,AAPL,GS,JPM,MSFT,XOM
Date,,,,,
2024-12-26,257.375580,566.623352,235.823608,432.973633,101.351990
2024-12-27,253.967392,561.700195,233.912888,425.482574,101.342468
2024-12-30,250.598892,559.136414,232.118576,419.849335,100.657204


### 2b · Volatility Forecasting (Rolling, EWMA, GARCH)

In [ ]:
from arch import arch_model

ticker = 'AAPL'
r      = rets[ticker] * 100          # scale for GARCH stability

# Rolling 21-day annualised vol
roll_vol = r.rolling(21).std() * np.sqrt(252) / 100

# EWMA (λ = 0.94 — RiskMetrics standard)
ewma_vol = r.ewm(span=21).std() * np.sqrt(252) / 100

# GARCH(1,1)
garch     = arch_model(r, vol='Garch', p=1, q=1, rescale=False)
garch_fit = garch.fit(disp='off')
garch_vol = garch_fit.conditional_volatility * np.sqrt(252) / 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_vol.index, y=roll_vol,  name='Rolling 21d'))
fig.add_trace(go.Scatter(x=ewma_vol.index, y=ewma_vol,  name='EWMA'))
fig.add_trace(go.Scatter(x=garch_vol.index, y=garch_vol, name='GARCH(1,1)'))
fig.update_layout(title=f'{ticker} — Annualised Volatility Forecasts',
                  yaxis_tickformat='.0%', template='plotly_dark')
fig.show()

print(f'Latest GARCH vol: {garch_vol.iloc[-1]:.2%}')

Latest GARCH vol: 21.04%


### 2c · VaR & CVaR Engine

In [ ]:
from scipy import stats

def compute_risk(returns: pd.Series, confidence_levels=(0.95, 0.99)):
    """Return DataFrame with Historical, Parametric and Monte-Carlo VaR/CVaR."""
    rows = []
    mu, sigma = returns.mean(), returns.std()
    sim = np.random.normal(mu, sigma, 100_000)

    for cl in confidence_levels:
        α = 1 - cl

        hist_var   = -np.percentile(returns, α * 100)
        hist_cvar  = -returns[returns <= -hist_var].mean()

        param_var  = -(mu + stats.norm.ppf(α) * sigma)
        param_cvar = -(mu - sigma * stats.norm.pdf(stats.norm.ppf(α)) / α)

        mc_var     = -np.percentile(sim, α * 100)
        mc_cvar    = -sim[sim <= -mc_var].mean()

        rows.append({
            'Confidence': f'{cl:.0%}',
            'Hist VaR':   f'{hist_var:.2%}',
            'Hist CVaR':  f'{hist_cvar:.2%}',
            'Param VaR':  f'{param_var:.2%}',
            'Param CVaR': f'{param_cvar:.2%}',
            'MC VaR':     f'{mc_var:.2%}',
            'MC CVaR':    f'{mc_cvar:.2%}',
        })
    return pd.DataFrame(rows)

display(compute_risk(rets['AAPL']))

,Confidence,Hist VaR,Hist CVaR,Param VaR,Param CVaR,MC VaR,MC CVaR
0,95%,3.01%,4.44%,3.16%,4.00%,3.16%,4.00%
1,99%,5.03%,7.03%,4.53%,5.20%,4.53%,5.21%


### 2d · Regime Detection (Hidden Markov Model)

In [ ]:
# pip install hmmlearn
from hmmlearn.hmm import GaussianHMM

r_arr = rets['AAPL'].values.reshape(-1, 1)

hmm = GaussianHMM(n_components=3, covariance_type='full',
                  n_iter=200, random_state=42)
hmm.fit(r_arr)
regimes = hmm.predict(r_arr)

# Label regimes by mean return (0=Bear, 1=Neutral, 2=Bull)
means   = {i: hmm.means_[i][0] for i in range(3)}
ranking = sorted(means, key=means.get)          # ascending mean → Bear first
labels  = {ranking[0]: 'Bear 🔴', ranking[1]: 'Neutral ⚪', ranking[2]: 'Bull 🟢'}
regime_labels = pd.Series([labels[r] for r in regimes], index=rets.index)

fig = px.scatter(x=rets.index, y=rets['AAPL'],
                 color=regime_labels,
                 color_discrete_map={'Bear 🔴': 'red', 'Neutral ⚪': 'grey', 'Bull 🟢': 'green'},
                 title='AAPL Daily Returns — HMM Regime Detection',
                 template='plotly_dark')
fig.update_traces(marker_size=3)
fig.show()

current_regime = regime_labels.iloc[-1]
print(f'Current regime: {current_regime}')

Current regime: Neutral ⚪


---
## Phase 3 — Portfolio Optimisation (Efficient Frontier)

In [ ]:
from scipy.optimize import minimize

mu_annual  = rets.mean() * 252
cov_annual = rets.cov()  * 252
n          = len(TICKERS)

# --- Sentiment boost: nudge expected returns with NLP score ---
# (Positive sentiment → +1 % to all tickers as a simple proxy)
sentiment_boost = portfolio_sentiment * 0.01
mu_ai = mu_annual + sentiment_boost
print(f'Sentiment boost applied: {sentiment_boost:+.4f} per ticker')

def portfolio_stats(w):
    ret  = w @ mu_ai
    vol  = np.sqrt(w @ cov_annual @ w)
    sharpe = (ret - RISK_FREE) / vol
    return ret, vol, sharpe

def neg_sharpe(w): return -portfolio_stats(w)[2]

constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
bounds      = [(0, 1)] * n
w0          = np.ones(n) / n

# Max-Sharpe portfolio
opt = minimize(neg_sharpe, w0, method='SLSQP',
               bounds=bounds, constraints=constraints)
w_sharpe = opt.x

# Efficient frontier (Monte Carlo random portfolios)
N_SIM = 3000
sim_w = np.random.dirichlet(np.ones(n), N_SIM)
sim_stats = np.array([portfolio_stats(w) for w in sim_w])   # (ret, vol, sharpe)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sim_stats[:, 1], y=sim_stats[:, 0],
    mode='markers',
    marker=dict(color=sim_stats[:, 2], colorscale='Viridis', size=4,
                showscale=True, colorbar=dict(title='Sharpe')),
    name='Simulated portfolios'
))
r_opt, v_opt, s_opt = portfolio_stats(w_sharpe)
fig.add_trace(go.Scatter(
    x=[v_opt], y=[r_opt], mode='markers+text',
    marker=dict(color='red', size=14, symbol='star'),
    text=['Max Sharpe'], textposition='top right', name='Max Sharpe'
))
fig.update_layout(title='Efficient Frontier (AI-enhanced returns)',
                  xaxis_title='Volatility', yaxis_title='Return',
                  xaxis_tickformat='.0%', yaxis_tickformat='.0%',
                  template='plotly_dark')
fig.show()

print('\n📌 Optimal weights (Max-Sharpe):')
for t, w in zip(TICKERS, w_sharpe):
    print(f'  {t}: {w:.1%}')
print(f'\n  Return: {r_opt:.2%}  |  Vol: {v_opt:.2%}  |  Sharpe: {s_opt:.2f}')

Sentiment boost applied: +0.0003 per ticker



📌 Optimal weights (Max-Sharpe):
  AAPL: 58.6%
  MSFT: 28.5%
  XOM: 0.0%
  GS: 7.3%
  JPM: 5.6%

  Return: 27.85%  |  Vol: 27.01%  |  Sharpe: 0.85


---
## Phase 4 — Regime-Aware Monte Carlo Simulator

In [ ]:
HORIZON   = 252          # trading days (~1 year)
N_PATHS   = 10_000
PORT_VAL  = 1_000_000    # starting portfolio value ($)

# Regime parameters (from HMM fits; adjust to your calibration)
REGIMES = {
    'Bull 🟢':    {'drift': 0.12/252,  'vol': 0.12/np.sqrt(252)},
    'Neutral ⚪': {'drift': 0.04/252,  'vol': 0.18/np.sqrt(252)},
    'Bear 🔴':    {'drift': -0.08/252, 'vol': 0.28/np.sqrt(252)},
}

# Use current regime
regime = REGIMES.get(current_regime, REGIMES['Neutral ⚪'])
drift  = regime['drift']
vol    = regime['vol']

# Sentiment overlay: positive sentiment nudges drift up
drift_adj = drift + portfolio_sentiment * 0.0002

# Simulate GBM paths
dt      = 1
shocks  = np.random.normal(0, 1, (HORIZON, N_PATHS))
log_ret = (drift_adj - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * shocks
paths   = PORT_VAL * np.exp(np.cumsum(log_ret, axis=0))
paths   = np.vstack([np.full(N_PATHS, PORT_VAL), paths])

# Summary stats
final = paths[-1]
var95 = np.percentile(final, 5)
cvar95 = final[final <= var95].mean()

# Plot a sample of paths + percentile bands
pct_5  = np.percentile(paths, 5,  axis=1)
pct_50 = np.percentile(paths, 50, axis=1)
pct_95 = np.percentile(paths, 95, axis=1)
days   = np.arange(HORIZON + 1)

fig = go.Figure()
# Show 200 random paths
for i in np.random.choice(N_PATHS, 200, replace=False):
    fig.add_trace(go.Scatter(x=days, y=paths[:, i],
                             line=dict(width=0.4, color='steelblue'),
                             showlegend=False))
fig.add_trace(go.Scatter(x=days, y=pct_95, name='95th pct',
                         line=dict(color='green', width=2)))
fig.add_trace(go.Scatter(x=days, y=pct_50, name='Median',
                         line=dict(color='white', width=2)))
fig.add_trace(go.Scatter(x=days, y=pct_5,  name='5th pct',
                         line=dict(color='red', width=2)))
fig.update_layout(
    title=f'Monte Carlo — {N_PATHS:,} paths | Regime: {current_regime}',
    xaxis_title='Trading Days', yaxis_title='Portfolio Value ($)',
    template='plotly_dark'
)
fig.show()

print(f'\n📉 1-Year Risk Summary (regime: {current_regime})')
print(f'  Median final value : ${pct_50[-1]:>12,.0f}')
print(f'  95th pct           : ${pct_95[-1]:>12,.0f}')
print(f'  5th  pct           : ${pct_5[-1]:>12,.0f}')
print(f'  VaR  95%           : ${PORT_VAL - var95:>12,.0f}  ({1 - var95/PORT_VAL:.2%} loss)')
print(f'  CVaR 95%           : ${PORT_VAL - cvar95:>12,.0f}  ({1 - cvar95/PORT_VAL:.2%} loss)')


📉 1-Year Risk Summary (regime: Neutral ⚪)
  Median final value : $   1,023,383
  95th pct           : $   1,364,444
  5th  pct           : $     761,755
  VaR  95%           : $     238,245  (23.82% loss)
  CVaR 95%           : $     292,322  (29.23% loss)


---
## Stress Test — What if oil drops 20%? / Rates rise 100 bps?


In [ ]:
SCENARIOS = {
    'Oil −20%':          {'drift_shock': -0.06/252, 'vol_mult': 1.4},
    'Rates +100 bps':    {'drift_shock': -0.03/252, 'vol_mult': 1.2},
    'Market crash −30%': {'drift_shock': -0.25/252, 'vol_mult': 2.5},
    'Base case':         {'drift_shock':  0.0,       'vol_mult': 1.0},
}

rows = []
for name, params in SCENARIOS.items():
    d = drift_adj + params['drift_shock']
    v = vol       * params['vol_mult']
    lr = (d - 0.5*v**2) + v * np.random.normal(0, 1, (HORIZON, 5000))
    f  = PORT_VAL * np.exp(lr.sum(axis=0))
    rows.append({
        'Scenario': name,
        'Median P&L':  f'${np.median(f) - PORT_VAL:>+,.0f}',
        '5th pct P&L': f'${np.percentile(f, 5) - PORT_VAL:>+,.0f}',
        'Prob of loss': f'{(f < PORT_VAL).mean():.1%}',
    })

display(pd.DataFrame(rows))

,Scenario,Median P&L,5th pct P&L,Prob of loss
0,Oil −20%,"$-43,783","$-362,334",57.1%
1,Rates +100 bps,"$-12,469","$-309,705",52.1%
2,Market crash −30%,"$-260,621","$-647,918",74.7%
3,Base case,"$+24,469","$-236,103",44.9%
